# ML용 농장현황 전처리

`감염농장_전처리.ipynb`를 먼저 같은 `PROVINCE`로 실행해서 `data/{PROVINCE}/농장현황/`, `data/{PROVINCE}/감염농장/`이 만들어져 있어야 한다 (이 노트북은 그 결과를 그대로 사용).

감염농장의 발생일(`OCCRRNC_DE`) 기준 **앞뒤 6개월**(예: 2017-01-15 발생 → 2016-07-15~2017-07-15)에 해당하는 농장현황만 모은다. 연도 단위로 자르면 1월 발생 건이 전년도 후반기 농장현황의 영향을 못 받게 되므로, 발생일 기준으로 정확히 ±6개월 윈도우를 잡아 연도 경계를 넘어가도 챙긴다. 감염농장이 여러 건이면 같은 윈도우가 중복으로 모이므로, 시군명·농장명·상세구분·소재지지번주소·위도·경도·사육두수가 같으면 중복 제거한다. **감염농장과 조인하지 않고, 농장현황 데이터만 그대로 저장한다.**

**출력**: `ML/ML_{PROVINCE}_농장현황.csv`, 건수 요약은 `ML/ML_농장현황_건수.csv`에 시/도별로 누적 기록

## 1. 시/도 선택 (이 변수만 바꿔서 재실행)

In [1]:
PROVINCE = "충청북도"  # 예: "경기도", "전라남도" — data/{PROVINCE}/ 폴더를 처리한다

## 2. 경로 설정

In [2]:
import glob
import os
import unicodedata
import pandas as pd

PROVINCE = unicodedata.normalize("NFC", PROVINCE)  # macOS NFD/NFC 정규화 (감염농장_전처리.ipynb와 동일한 이유)

# 이 노트북은 ML/ 폴더에 있고, Jupyter/nbconvert 모두 노트북 파일이 있는 폴더를 작업 디렉터리로 잡으므로
# repo 루트의 data/는 한 단계 위(..)에서 찾는다.
CENSUS_DIR = os.path.join("..", "data", PROVINCE, "농장현황")
INFECTION_DIR = os.path.join("..", "data", PROVINCE, "감염농장")

assert os.path.isdir(CENSUS_DIR) and os.path.isdir(INFECTION_DIR), (
    f"{CENSUS_DIR} 또는 {INFECTION_DIR}가 없습니다. "
    f"먼저 감염농장_전처리.ipynb를 PROVINCE='{PROVINCE}'로 실행하세요."
)
print("농장현황 폴더:", CENSUS_DIR)
print("감염농장 폴더:", INFECTION_DIR)

농장현황 폴더: ../data/충청북도/농장현황
감염농장 폴더: ../data/충청북도/감염농장


## 3. 발생일 기준 ±6개월 농장현황 윈도우 + 중복 제거 (감염농장과 조인하지 않음)

In [3]:
DEDUP_COLS = ["시군명", "농장명", "상세구분", "소재지지번주소", "WGS84위도", "WGS84경도", "사육두수(마리)"]

sigun_frames = []

for infection_file in sorted(glob.glob(os.path.join(INFECTION_DIR, "*_감염농장.csv"))):
    sigun = os.path.splitext(os.path.basename(infection_file))[0].replace("_감염농장", "")
    census_file = os.path.join(CENSUS_DIR, f"{sigun}_농장현황.csv")

    if not os.path.exists(census_file):
        print(f"  ⚠️  {sigun}: 농장현황 파일 없음, 스킵")
        continue

    infection_df = pd.read_csv(infection_file, encoding="utf-8-sig")
    census_df = pd.read_csv(census_file, encoding="utf-8-sig")
    census_df["조사날짜"] = pd.to_datetime(census_df["조사날짜"])

    # 이 시군에서 감염이 발생한 날짜들만 (감염농장이 여러 건이어도 같은 날짜면 한 번만 처리)
    occr_dates = sorted(pd.to_datetime(infection_df["OCCRRNC_DE"], format="%Y%m%d", errors="coerce").dropna().unique())

    for occr_date in occr_dates:
        occr_date = pd.Timestamp(occr_date)
        start = occr_date - pd.DateOffset(months=6)
        end = occr_date + pd.DateOffset(months=6)
        windowed = census_df[(census_df["조사날짜"] >= start) & (census_df["조사날짜"] <= end)]
        before = len(windowed)
        windowed = windowed.drop_duplicates(subset=DEDUP_COLS, keep="first")
        after = len(windowed)
        sigun_frames.append(windowed)
        print(f"  {sigun} {occr_date.date()} ±6개월 ({start.date()}~{end.date()}): {before} → {after}건")

census_window_df = pd.concat(sigun_frames, ignore_index=True) if sigun_frames else pd.DataFrame()
print(f"\n✓ 윈도우 합계: {len(census_window_df)}건")

  ⚠️  괴산군: 농장현황 파일 없음, 스킵
  ⚠️  영동군: 농장현황 파일 없음, 스킵
  ⚠️  옥천군: 농장현황 파일 없음, 스킵
  음성군 2003-12-12 ±6개월 (2003-06-12~2004-06-12): 0 → 0건
  음성군 2003-12-17 ±6개월 (2003-06-17~2004-06-17): 0 → 0건
  음성군 2003-12-19 ±6개월 (2003-06-19~2004-06-19): 0 → 0건
  음성군 2003-12-24 ±6개월 (2003-06-24~2004-06-24): 0 → 0건
  음성군 2014-02-07 ±6개월 (2013-08-07~2014-08-07): 0 → 0건
  음성군 2014-02-10 ±6개월 (2013-08-10~2014-08-10): 0 → 0건
  음성군 2014-02-19 ±6개월 (2013-08-19~2014-08-19): 0 → 0건
  음성군 2014-02-20 ±6개월 (2013-08-20~2014-08-20): 0 → 0건
  음성군 2014-02-21 ±6개월 (2013-08-21~2014-08-21): 0 → 0건
  음성군 2014-02-23 ±6개월 (2013-08-23~2014-08-23): 0 → 0건
  음성군 2014-02-24 ±6개월 (2013-08-24~2014-08-24): 0 → 0건
  음성군 2014-02-25 ±6개월 (2013-08-25~2014-08-25): 0 → 0건
  음성군 2014-02-28 ±6개월 (2013-08-28~2014-08-28): 0 → 0건
  음성군 2015-02-24 ±6개월 (2014-08-24~2015-08-24): 0 → 0건
  음성군 2015-02-25 ±6개월 (2014-08-25~2015-08-25): 0 → 0건
  음성군 2015-02-26 ±6개월 (2014-08-26~2015-08-26): 0 → 0건
  음성군 2015-03-04 ±6개월 (2014-09-04~2015-09-04): 0 → 0건
  음성

## 4. 시군 간 최종 중복 제거 후 저장 → `ML/ML_{PROVINCE}_농장현황.csv`, 건수 요약 누적

In [4]:
before_total = len(census_window_df)
final_df = census_window_df.drop_duplicates(subset=DEDUP_COLS, keep="first").reset_index(drop=True)

## 5. 닭/오리 세부분류 정규화 + 컬럼 순서 통일

시/도마다 `축종명`에 세부 항목(예: `육계`, `종계`, `육용오리`)이 그대로 들어있거나 `상세구분`에 들어있는 등 형식이 다르다. `livestock_codes.csv`(정부 API의 `LVSTCKSPC_NM`에서 파생된 표준 코드표) 기준으로, 닭/오리 관련 단어가 `축종명`이나 `상세구분`에 있으면 `축종명`을 닭/오리로, `상세구분`을 표준 세부값으로 통일한다.

- `종계/산란계`처럼 한 행에 표준 세부값이 두 개 들어있으면 둘 다 보존해서 `/`로 합친다 (임의로 하나를 버리지 않음).
- 닭/오리 둘 다 섞였거나 닭·오리가 아닌 다른 축종(꿩·타조·산양 등)과 섞인 행은 원본 데이터 자체의 모순(예: 꿩 농장인데 `상세구분`에 산란계가 적힌 경우)이라 임의로 고치지 않고 원본 값 그대로 둔다.
- 컬럼 순서를 `시군명, 농장명, 축종명, 상세구분, 사육두수(마리), 소재지지번주소, WGS84위도, WGS84경도, 조사날짜` 9개로 통일한다 (경기도에만 있는 소재지우편번호/소재지도로명주소/데이터기준일자/비고는 제외).

In [5]:
import re

STANDARD_COLS = ["시군명", "농장명", "축종명", "상세구분", "사육두수(마리)", "소재지지번주소", "WGS84위도", "WGS84경도", "조사날짜"]

# livestock_codes.csv의 닭/오리 상세구분 어휘 기준, 구체적인 단어를 먼저 검사한다 (예: "산란중추"를 "산란계"보다 먼저 검사)
CHICKEN_DETAIL_PATTERNS = [
    ("토종닭종계", "종계"),
    ("육용종계", "육용종계"),
    ("산란종계", "산란종계"),
    ("산란중추", "산란중추"),
    ("산란육성계", "산란중추"),
    ("산란계", "산란계"),
    ("원종계", "원종계"),
    ("종계", "종계"),
    ("토종닭", "토종닭"),
    ("오골계", "토종닭"),
    ("백세미", "백세미"),
    ("삼계", "육계"),
    ("육계", "육계"),
    ("육닭", "육계"),
    ("관상계", "기타"),
    ("일괄", "일괄"),
    ("비분류", "비분류"),
]
DUCK_DETAIL_PATTERNS = [
    ("종오리", "종오리"),
    ("육용오리", "육용오리"),
    ("산란오리", "산란오리"),
    ("페킹덕", "페킹덕"),
]


def classify_token(tok):
    tok = re.sub(r"\([^)]*\)", "", tok).strip()  # "닭(부화업)" 같은 괄호 설명 제거
    if not tok:
        return None
    for pat, detail in DUCK_DETAIL_PATTERNS:
        if pat in tok:
            return ("오리", detail)
    if "오리" in tok:
        return ("오리", None)
    for pat, detail in CHICKEN_DETAIL_PATTERNS:
        if pat in tok:
            return ("닭", detail)
    if "닭" in tok:
        return ("닭", None)
    return ("기타", tok)


def classify_poultry(axis_raw, detail_raw):
    """축종명/상세구분 원본 값을 받아 (새 축종명, 새 상세구분)을 반환한다.
    닭/오리가 아닌 축종이거나, 닭·오리가 섞이거나 다른 축종과 섞인 모호한 행은 원본 그대로 반환한다."""
    parts = []
    if isinstance(axis_raw, str):
        parts.append(axis_raw)
    if isinstance(detail_raw, str) and detail_raw != axis_raw:
        parts.append(detail_raw)
    if not parts:
        return (axis_raw, detail_raw)

    tokens = [t.strip() for p in parts for t in re.split(r"[,/+]", p) if t.strip()]

    species_set, chicken_details, duck_details, has_other = set(), [], [], False
    for tok in tokens:
        c = classify_token(tok)
        if c is None:
            continue
        sp, detail = c
        species_set.add(sp)
        if sp == "닭" and detail and detail not in chicken_details:
            chicken_details.append(detail)
        if sp == "오리" and detail and detail not in duck_details:
            duck_details.append(detail)
        if sp == "기타":
            has_other = True

    poultry_species = species_set & {"닭", "오리"}
    if not poultry_species or len(poultry_species) > 1 or has_other:
        return (axis_raw, detail_raw)  # 닭/오리 무관 또는 모호 -> 원본 유지

    if "닭" in poultry_species:
        if "종계" in chicken_details and ("육용종계" in chicken_details or "산란종계" in chicken_details):
            chicken_details.remove("종계")  # 육용종계/산란종계가 더 구체적이므로 일반 종계는 버림
        return ("닭", "/".join(chicken_details) if chicken_details else pd.NA)
    return ("오리", "/".join(duck_details) if duck_details else pd.NA)


if "축종명" not in final_df.columns:
    final_df["축종명"] = pd.NA
if "상세구분" not in final_df.columns:
    final_df["상세구분"] = pd.NA

normalized = [classify_poultry(a, d) for a, d in zip(final_df["축종명"], final_df["상세구분"])]
final_df["축종명"] = [n[0] for n in normalized]
final_df["상세구분"] = [n[1] for n in normalized]
final_df = final_df[STANDARD_COLS]

print("✓ 닭/오리 정규화 + 컬럼 순서 통일 완료")
print("축종명 분포:", final_df["축종명"].value_counts(dropna=False).to_dict())

✓ 닭/오리 정규화 + 컬럼 순서 통일 완료
축종명 분포: {'닭': 171, '오리': 79, '메추리': 6, '타조': 1, '면양, 타조, 염소': 1, '육계, 산양': 1, '한우, 메추리': 1, '꿩': 1}


## 6. 저장 → `ML/ML_{PROVINCE}_농장현황.csv`, 건수 요약 누적

In [ ]:
out_path = f"ML_{PROVINCE}_농장현황.csv"  # 노트북이 이미 ML/ 안에 있으므로 그대로 저장
final_df.to_csv(out_path, index=False, encoding="utf-8")

count_path = "ML_농장현황_건수.csv"
if os.path.exists(count_path):
    count_df = pd.read_csv(count_path, encoding="utf-8-sig")
    count_df = count_df[count_df["시도"] != PROVINCE]
else:
    count_df = pd.DataFrame(columns=["시도", "건수"])

new_row = pd.DataFrame([{"시도": PROVINCE, "건수": len(final_df)}])
count_df = pd.concat([count_df, new_row], ignore_index=True)
count_df.to_csv(count_path, index=False, encoding="utf-8")

print(f"✓ 저장 완료: {out_path}")
print(f"  - 시군/연도별 윈도우 합계 {before_total}건 → 시군 간 중복 제거 후 {len(final_df)}건")
print(f"✓ 건수 요약 갱신: {count_path}")
print(count_df.to_string(index=False))